# 01e Parse ERA5 Long-Run Climatology

This notebook builds long-run ERA5 climatology inputs for the h=3-h=10 synthetic forecast emulator. It uses pre-sales-period weather only, with parameter aggregates restricted to local dates up to and including 2021-12-31. ERA5 is used for climatology centres, empirical bounds, wet-day and precipitation-tail rates, cloud-regime priors, pressure/MSLP regime calibration, and audit diagnostics. ERA5 variables, including pressure/MSLP, are not direct ML demand features.


## Pipeline position, inputs, and method contract

This notebook follows notebooks 01a-01d and feeds notebook 04 through long-run ERA5 climatology and pressure-regime priors. It reads raw ERA5 timeseries NetCDF files under `data/era5/raw/timeseries/store_point_XXX/` and store-point metadata from `data/era5/metadata/era5_store_points_from_cloud_points.csv`, which links `store_point_XXX` to `AvdelingID` where available.

When executed, the notebook writes `data/processed/era5_longrun_weather_windows.parquet`, `data/processed/era5_longrun_climatology_windows.parquet`, `data/processed/era5_longrun_climatology_parameters.parquet`, `data/processed/era5_longrun_raw_file_audit.parquet`, `data/processed/era5_longrun_climatology_audit.parquet`, and CSV summaries under `outputs/era5_longrun_climatology/`.

Aggregation windows are defined in Europe/Oslo local time: `full_day_00_23`, `trade_08_18`, and `trade_08_19`, with `trade_08_19` as the main thesis trading window. Unit conversions are explicit: `t2m` and `d2m` from Kelvin to Celsius; `u10` and `v10` to scalar `wind_ms`; `tp` from metres to mm with tiny negatives clipped to zero; `msl` from Pa to hPa; `tcc` from fraction to percent; and relative humidity from temperature and dew point using the Magnus/Tetens approximation.

The leakage boundary is enforced by using only rows with local date `<= 2021-12-31` for parameter aggregates. Each parameter group records `source_period_start`, `source_period_end`, and `max_source_date` for downstream assertions. Pressure-regime thresholds are estimated within the parameter group using p20/p80 pressure levels and p75/p90 absolute pressure-change thresholds. Regime classes are `stable_high`, `low_unsettled`, `neutral`, and `transition`, with transition overriding the pressure level when absolute pressure change is high.


## Imports and paths

The setup defines project-root discovery, ERA5 raw and processed paths, output folders, window definitions, and shared constants for local-time aggregation.


In [ ]:
import re
import time
from pathlib import Path

import netCDF4
import numpy as np
import pandas as pd
from netCDF4 import num2date

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / m).exists() for m in MARKER_FILES):
            return candidate
    raise FileNotFoundError("project root not found")


PROJECT_ROOT = find_project_root()
RAW_TS_DIR = PROJECT_ROOT / "data" / "era5" / "raw" / "timeseries"
META_CSV = PROJECT_ROOT / "data" / "era5" / "metadata" / "era5_store_points_from_cloud_points.csv"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUT_DIR = PROJECT_ROOT / "outputs" / "era5_longrun_climatology"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

ERA5_VARIABLES = ("t2m", "d2m", "u10", "v10", "tp", "msl", "tcc")
MAX_LOCAL_DATE = pd.Timestamp("2021-12-31")
LOCAL_TZ = "Europe/Oslo"
WET_THRESHOLD_MM = 0.1

AGG_WINDOWS = {
    "full_day_00_23": (0, 23),
    "trade_08_18": (8, 18),
    "trade_08_19": (8, 19),
}

FILENAME_RE = re.compile(
    r"^store_point_(?P<sid>\d{3})_(?P<var>[a-z0-9]+)_(?P<y1>\d{4})_(?P<y2>\d{4})$"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw timeseries dir: {RAW_TS_DIR}")
print(f"Max source date: {MAX_LOCAL_DATE.date()}")


## Raw ERA5 audit

This step walks the raw ERA5 timeseries tree, registers each year chunk, reads file-level metadata, and writes audit outputs. It inspects time span, units, and coordinates without loading full time series into memory.


In [ ]:
store_dirs = sorted(p for p in RAW_TS_DIR.iterdir() if p.is_dir() and p.name.startswith("store_point_"))
print(f"Store-point folders: {len(store_dirs)} (expected 46)")

records, schema_samples = [], []
schema_targets = {(f"store_point_{i:03d}", v) for i in (1, 12, 24, 36, 46) for v in ERA5_VARIABLES}
t0 = time.time()
for store_dir in store_dirs:
    sid = store_dir.name
    zips = sorted(store_dir.glob("*.zip"))
    for zip_path in zips:
        m = FILENAME_RE.match(zip_path.stem)
        if not m:
            continue
        var, y1, y2 = m.group("var"), int(m.group("y1")), int(m.group("y2"))
        ext_dir = store_dir / zip_path.stem
        ncs = sorted(ext_dir.glob("*.nc")) if ext_dir.exists() else []
        rec = {
            "store_point": sid, "variable": var, "y1": y1, "y2": y2,
            "kind": "chunk", "name": zip_path.stem,
            "size_bytes": zip_path.stat().st_size,
            "has_extracted_dir": ext_dir.exists(),
            "has_netcdf": bool(ncs),
            "netcdf_path": ncs[0].relative_to(PROJECT_ROOT).as_posix() if ncs else None,
            "n_timesteps": None, "time_min": None, "time_max": None,
            "var_units": None, "var_long_name": None,
            "lat": None, "lon": None, "error": None,
        }
        if ncs:
            try:
                with netCDF4.Dataset(ncs[0], "r") as ds:
                    t = ds.variables.get("valid_time") or ds.variables.get("time")
                    rec["n_timesteps"] = int(t.shape[0])
                    times = num2date(t[:1], t.units, calendar=getattr(t, "calendar", "standard"))
                    rec["time_min"] = pd.Timestamp(str(times[0]))
                    times = num2date(t[-1:], t.units, calendar=getattr(t, "calendar", "standard"))
                    rec["time_max"] = pd.Timestamp(str(times[0]))
                    if var in ds.variables:
                        v = ds.variables[var]
                        rec["var_units"] = getattr(v, "units", None)
                        rec["var_long_name"] = getattr(v, "long_name", None)
                    for coord in ("latitude", "longitude"):
                        if coord in ds.variables and ds.variables[coord].size:
                            try:
                                rec[coord[:3]] = float(ds.variables[coord][...])
                            except Exception:
                                pass
                    if (sid, var) in schema_targets:
                        schema_samples.append({**rec, "year_chunk": f"{y1}_{y2}"})
                        schema_targets.discard((sid, var))
            except Exception as exc:
                rec["error"] = f"{type(exc).__name__}: {exc}"
        records.append(rec)
elapsed = time.time() - t0

audit_df = pd.DataFrame(records)
for c in ("time_min", "time_max"):
    audit_df[c] = pd.to_datetime(audit_df[c], errors="coerce")
audit_df.to_parquet(
    PROCESSED / "era5_longrun_raw_file_audit.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy",
)
pd.DataFrame(schema_samples).to_csv(OUT_DIR / "era5_raw_schema_samples.csv", index=False)

print(f"Scanned {len(records)} chunk files in {elapsed:.1f}s")
print(f"  stores covered: {audit_df['store_point'].nunique()}")
print(f"  variables: {sorted(audit_df['variable'].dropna().unique())}")
print(f"  errors: {int(audit_df['error'].notna().sum())}")


## Hourly parsing and local-window aggregation

This step reads each store-variable timeseries, joins variables, converts units, maps timestamps to Europe/Oslo local time, and aggregates to daily `aggregation_window` rows. Each store contributes daily-window records for the available ERA5 history.


In [ ]:
def read_one_chunk(nc_path: Path, varname: str) -> pd.DataFrame:
    with netCDF4.Dataset(nc_path, "r") as ds:
        t = ds.variables.get("valid_time") or ds.variables.get("time")
        if t is None or varname not in ds.variables:
            raise ValueError(f"missing time or variable in {nc_path}")
        times = num2date(
            t[:],
            t.units,
            calendar=getattr(t, "calendar", "standard"),
            only_use_cftime_datetimes=False,
            only_use_python_datetimes=True,
        )
        values = np.asarray(ds.variables[varname][:], dtype="float32")
    return pd.DataFrame({"utc": pd.to_datetime(times), varname: values})


def parse_store(store_dir: Path) -> tuple[pd.DataFrame, dict]:
    store_id = store_dir.name
    diag = {"store_point": store_id, "lat": None, "lon": None}

    var_frames = {}
    for var in ERA5_VARIABLES:
        chunk_dirs = sorted(
            d for d in store_dir.iterdir()
            if d.is_dir()
            and FILENAME_RE.match(d.name)
            and FILENAME_RE.match(d.name).group("var") == var
        )
        if not chunk_dirs:
            continue
        parts = []
        for cd in chunk_dirs:
            ncs = sorted(cd.glob("*.nc"))
            if not ncs:
                continue
            try:
                parts.append(read_one_chunk(ncs[0], var))
                if diag["lat"] is None:
                    with netCDF4.Dataset(ncs[0], "r") as ds:
                        if "latitude" in ds.variables and ds.variables["latitude"].size:
                            try:
                                diag["lat"] = float(ds.variables["latitude"][...])
                            except Exception:
                                pass
                        if "longitude" in ds.variables and ds.variables["longitude"].size:
                            try:
                                diag["lon"] = float(ds.variables["longitude"][...])
                            except Exception:
                                pass
            except Exception as exc:
                print(f"  WARN {store_id} {var} {cd.name}: {type(exc).__name__}: {exc}")
        if parts:
            var_frames[var] = pd.concat(parts, ignore_index=True).drop_duplicates(subset="utc")

    if not var_frames:
        return pd.DataFrame(), diag

    wide = None
    for var, df in var_frames.items():
        wide = df.copy() if wide is None else wide.merge(df, on="utc", how="outer")
    wide = wide.sort_values("utc").reset_index(drop=True)

    # Convert ERA5 units to modelling units before aggregation.
    if "t2m" in wide.columns:
        wide["temp_c"] = wide["t2m"].astype("float32") - 273.15
    if "d2m" in wide.columns:
        wide["dewpoint_c"] = wide["d2m"].astype("float32") - 273.15
    if "u10" in wide.columns and "v10" in wide.columns:
        wide["wind_ms"] = np.hypot(
            wide["u10"].astype("float32"),
            wide["v10"].astype("float32"),
        ).astype("float32")
    if "tp" in wide.columns:
        wide["precip_mm"] = (wide["tp"].astype("float32") * 1000.0).astype("float32")
        wide.loc[wide["precip_mm"] < 0, "precip_mm"] = 0.0
    if "msl" in wide.columns:
        wide["pressure_hpa"] = (wide["msl"].astype("float32") / 100.0).astype("float32")
    if "tcc" in wide.columns:
        wide["cloud_pct"] = np.clip(
            wide["tcc"].astype("float32") * 100.0,
            0.0,
            100.0,
        ).astype("float32")
    if "temp_c" in wide.columns and "dewpoint_c" in wide.columns:
        a, b = 17.625, 243.04
        T, Td = wide["temp_c"].to_numpy("float64"), wide["dewpoint_c"].to_numpy("float64")
        wide["humid_pct"] = np.clip(
            100.0 * np.exp(a * Td / (b + Td)) / np.exp(a * T / (b + T)),
            0.0,
            100.0,
        ).astype("float32")

    wide["utc"] = pd.to_datetime(wide["utc"])
    if wide["utc"].dt.tz is None:
        wide["utc"] = wide["utc"].dt.tz_localize("UTC")
    wide["local"] = wide["utc"].dt.tz_convert(LOCAL_TZ)
    wide["local_date"] = wide["local"].dt.date
    wide["local_hour"] = wide["local"].dt.hour
    wide = wide.loc[wide["local"].dt.normalize().dt.tz_localize(None) <= MAX_LOCAL_DATE].copy()

    derived = [c for c in ["temp_c", "dewpoint_c", "humid_pct", "wind_ms",
                           "precip_mm", "cloud_pct", "pressure_hpa"] if c in wide.columns]

    rows = []
    for window, (h_lo, h_hi) in AGG_WINDOWS.items():
        mask = (wide["local_hour"] >= h_lo) & (wide["local_hour"] <= h_hi)
        sub = wide.loc[mask, ["local_date", "local_hour", *derived]]
        if sub.empty:
            continue
        aggs = {}
        if "temp_c" in derived:
            aggs["temp_c"] = ["mean", "min", "max"]
        if "dewpoint_c" in derived:
            aggs["dewpoint_c"] = ["mean"]
        if "humid_pct" in derived:
            aggs["humid_pct"] = ["mean"]
        if "wind_ms" in derived:
            aggs["wind_ms"] = ["mean", "max"]
        if "precip_mm" in derived:
            aggs["precip_mm"] = ["sum"]
        if "cloud_pct" in derived:
            aggs["cloud_pct"] = ["mean"]
        if "pressure_hpa" in derived:
            aggs["pressure_hpa"] = ["mean", "min", "max"]
        grouped = sub.groupby("local_date").agg(aggs)
        grouped.columns = ["_".join(c) for c in grouped.columns]
        if "pressure_hpa" in derived:
            first = sub.groupby("local_date").apply(
                lambda g: g.sort_values("local_hour")["pressure_hpa"].iloc[0] if len(g) else np.nan,
                include_groups=False,
            )
            last = sub.groupby("local_date").apply(
                lambda g: g.sort_values("local_hour")["pressure_hpa"].iloc[-1] if len(g) else np.nan,
                include_groups=False,
            )
            tendency = (last - first).astype("float32")
            grouped["pressure_change_window"] = tendency
            grouped["pressure_abs_change_window"] = tendency.abs()
        obs_hours = sub.groupby("local_date")["local_hour"].nunique()
        expected = h_hi - h_lo + 1
        grouped["observed_hours"] = obs_hours.astype("int16")
        grouped["expected_hours_in_window"] = np.int16(expected)
        grouped["coverage_share"] = (grouped["observed_hours"] / float(expected)).astype("float32")
        grouped["is_partial_window"] = (grouped["observed_hours"] < expected).astype("int8")
        grouped["is_full_window"] = (grouped["observed_hours"] >= expected).astype("int8")
        grouped["aggregation_window"] = window
        grouped["window_first_hour_local"] = h_lo
        grouped["window_last_hour_local"] = h_hi
        grouped["store_point"] = store_id
        rows.append(grouped.reset_index())

    if not rows:
        return pd.DataFrame(), diag
    out = pd.concat(rows, ignore_index=True)
    out = out.rename(columns={
        "temp_c_mean": "temp_hist_mean",
        "temp_c_min": "temp_hist_min",
        "temp_c_max": "temp_hist_max",
        "dewpoint_c_mean": "dewpoint_hist_mean",
        "humid_pct_mean": "humid_hist_mean",
        "wind_ms_mean": "wind_hist_mean",
        "wind_ms_max": "wind_hist_max",
        "precip_mm_sum": "precip_hist_sum",
        "cloud_pct_mean": "cloud_hist_mean",
        "pressure_hpa_mean": "pressure_hist_mean",
        "pressure_hpa_min": "pressure_hist_min",
        "pressure_hpa_max": "pressure_hist_max",
        "pressure_change_window": "pressure_hist_change_window",
        "pressure_abs_change_window": "pressure_hist_abs_change_window",
        "local_date": "date",
    })
    out["date"] = pd.to_datetime(out["date"])
    out["window_timezone"] = LOCAL_TZ
    out["window_basis"] = "local_time"
    out["source"] = "era5_single_levels_timeseries"
    out["source_period_start"] = pd.to_datetime(wide["utc"].min().date())
    out["source_period_end"] = pd.to_datetime(wide["utc"].max().date())
    out["max_source_date"] = MAX_LOCAL_DATE.normalize()

    diag["n_rows"] = int(len(out))
    diag["date_min"] = out["date"].min()
    diag["date_max"] = out["date"].max()
    return out, diag


all_frames, all_diag = [], []
t0 = time.time()
for i, sd in enumerate(store_dirs, 1):
    f, diag = parse_store(sd)
    all_diag.append(diag)
    if not f.empty:
        all_frames.append(f)
    if i % 5 == 0 or i == len(store_dirs):
        print(
            f"  [{i:2d}/{len(store_dirs)}] {sd.name}: "
            f"rows={diag.get('n_rows', 0):>7}  "
            f"date={diag.get('date_min')}->{diag.get('date_max')}"
        )
big = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
print(f"\nConcatenated rows: {len(big):,} in {(time.time() - t0) / 60:.1f} min")

# Retain AvdelingID when metadata provides the anonymised store identifier.
if META_CSV.exists():
    meta = pd.read_csv(META_CSV)
    if "store_point" in meta.columns and "AvdelingID" in meta.columns:
        big = big.merge(
            meta[["store_point", "AvdelingID"]].drop_duplicates(),
            on="store_point",
            how="left",
        )

key_cols = [
    c for c in [
        "date",
        "store_point",
        "AvdelingID",
        "aggregation_window",
        "window_first_hour_local",
        "window_last_hour_local",
        "window_timezone",
        "window_basis",
        "source",
        "source_period_start",
        "source_period_end",
        "max_source_date",
        "expected_hours_in_window",
        "observed_hours",
        "coverage_share",
        "is_full_window",
        "is_partial_window",
    ]
    if c in big.columns
]
big = big[key_cols + [c for c in big.columns if c not in key_cols]]

WINDOWS_PATH = PROCESSED / "era5_longrun_weather_windows.parquet"
big.to_parquet(WINDOWS_PATH, index=False, engine="pyarrow", compression="snappy")
pd.DataFrame(all_diag).to_parquet(
    PROCESSED / "era5_longrun_climatology_audit.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy",
)
print(f"Wrote {WINDOWS_PATH}  ({WINDOWS_PATH.stat().st_size/1_000_000:.1f} MB)")


## Climatology parameters

This step builds the fallback hierarchy from full-window rows with `date <= 2021-12-31`. Parameter groups contain climatology means, empirical percentiles, wet-day and precipitation-tail summaries, cloud-regime shares, and pressure-regime diagnostics for downstream synthetic-emulator calibration.


In [ ]:
big = pd.read_parquet(PROCESSED / "era5_longrun_weather_windows.parquet")
big = big.loc[pd.to_datetime(big["date"]) <= MAX_LOCAL_DATE].copy()
big["date"] = pd.to_datetime(big["date"])
big["month"] = big["date"].dt.month.astype("int8")
def to_season(m):
    return {
        12: "winter",
        1: "winter",
        2: "winter",
        3: "spring",
        4: "spring",
        5: "spring",
        6: "summer",
        7: "summer",
        8: "summer",
    }.get(int(m), "autumn")

big["season"] = big["month"].apply(to_season).astype("string")
clim = big.loc[big["is_full_window"] == 1].copy()
clim.to_parquet(
    PROCESSED / "era5_longrun_climatology_windows.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy",
)
print(f"Climatology windows rows: {len(clim):,}")

def percentiles(s, ps=(0.05, 0.10, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99)):
    s = s.dropna()
    if len(s) == 0:
        return {f"p{int(p * 100):02d}": np.nan for p in ps}
    return {f"p{int(p * 100):02d}": float(s.quantile(p)) for p in ps}

def aggregate_group(g):
    rec = {"n_obs": int(len(g))}
    if "temp_hist_mean" in g.columns:
        s = g["temp_hist_mean"]
        rec["temp_clim_mean"] = float(s.mean())
        rec["temp_clim_min"] = float(s.min())
        rec["temp_clim_max"] = float(s.max())
        for k, v in percentiles(s, (0.05, 0.10, 0.5, 0.9, 0.95)).items():
            rec[f"temp_clim_{k}"] = v
    if "wind_hist_mean" in g.columns:
        s = g["wind_hist_mean"]
        rec["wind_clim_mean"] = float(s.mean())
        for k, v in percentiles(s, (0.5, 0.9, 0.95, 0.99)).items():
            rec[f"wind_clim_{k}"] = v
        rec["wind_empirical_cap"] = float(s.quantile(0.999))
    if "humid_hist_mean" in g.columns:
        s = g["humid_hist_mean"]
        rec["humid_clim_mean"] = float(s.mean())
        for k, v in percentiles(s, (0.1, 0.5, 0.9)).items():
            rec[f"humid_clim_{k}"] = v
    if "cloud_hist_mean" in g.columns:
        s = g["cloud_hist_mean"]
        rec["cloud_clim_mean"] = float(s.mean())
        for k, v in percentiles(s, (0.1, 0.5, 0.9)).items():
            rec[f"cloud_clim_{k}"] = v
        rec["cloud_open_share"] = float((s <= 25).mean())
        rec["cloud_partly_share"] = float(((s > 25) & (s < 75)).mean())
        rec["cloud_overcast_share"] = float((s >= 75).mean())
    if "precip_hist_sum" in g.columns:
        s = g["precip_hist_sum"]
        rec["precip_clim_mean"] = float(s.mean())
        for k, v in percentiles(s, (0.5, 0.9, 0.95, 0.99)).items():
            rec[f"precip_clim_{k}"] = v
        rec["precip_empirical_cap"] = float(s.quantile(0.999))
        rec["wet_day_share"] = float((s > WET_THRESHOLD_MM).mean())
        wet = s[s > WET_THRESHOLD_MM]
        for k, v in percentiles(wet, (0.5, 0.9, 0.95, 0.99)).items():
            rec[f"precip_wet_amount_{k}"] = v
    if "pressure_hist_mean" in g.columns:
        s = g["pressure_hist_mean"]
        rec["pressure_clim_mean_hpa"] = float(s.mean())
        for k, v in percentiles(s, (0.1, 0.5, 0.9)).items():
            rec[f"pressure_clim_{k}"] = v
        low_thr = float(s.quantile(0.20))
        high_thr = float(s.quantile(0.80))
        rec["pressure_low_threshold"] = low_thr
        rec["pressure_high_threshold"] = high_thr
        if "pressure_hist_abs_change_window" in g.columns:
            ch = g["pressure_hist_abs_change_window"]
            rec["pressure_abs_change_p75"] = float(ch.quantile(0.75))
            rec["pressure_abs_change_p90"] = float(ch.quantile(0.90))
            ch_thr_low, ch_thr_high = rec["pressure_abs_change_p75"], rec["pressure_abs_change_p90"]
            stable_high = (s >= high_thr) & (ch < ch_thr_high)
            low_unsettled = (s <= low_thr) & (ch >= ch_thr_low)
            transition = ch >= ch_thr_high
            stable_high = stable_high & ~transition
            low_unsettled = low_unsettled & ~transition
            neutral = ~(stable_high | low_unsettled | transition)
            rec["pressure_regime_share_stable_high"] = float(stable_high.mean())
            rec["pressure_regime_share_low_unsettled"] = float(low_unsettled.mean())
            rec["pressure_regime_share_neutral"] = float(neutral.mean())
            rec["pressure_regime_share_transition"] = float(transition.mean())
    rec["source_period_start"] = (
        pd.to_datetime(g["source_period_start"].min())
        if "source_period_start" in g.columns and len(g)
        else pd.NaT
    )
    rec["source_period_end"] = (
        pd.to_datetime(g["source_period_end"].max())
        if "source_period_end" in g.columns and len(g)
        else pd.NaT
    )
    rec["max_source_date"] = pd.to_datetime(g["date"].max()) if len(g) else pd.NaT
    return rec

def build_level(level, group_cols):
    rows = []
    if group_cols:
        for keys, g in clim.groupby(group_cols, observed=True):
            if not isinstance(keys, tuple):
                keys = (keys,)
            rec = aggregate_group(g)
            for k, v in zip(group_cols, keys):
                rec[k] = v
            rec["fallback_level"] = level
            rows.append(rec)
    else:
        rec = aggregate_group(clim)
        rec["fallback_level"] = level
        rows.append(rec)
    return pd.DataFrame(rows)

levels = [
    (1, ["store_point", "month", "aggregation_window"]),
    (2, ["store_point", "season", "aggregation_window"]),
    (3, ["month", "aggregation_window"]),
    (4, ["season", "aggregation_window"]),
    (5, ["aggregation_window"]),
]
params = pd.concat([build_level(lvl, cols) for lvl, cols in levels], ignore_index=True)
if "AvdelingID" in big.columns:
    avd_map = big[["store_point", "AvdelingID"]].drop_duplicates()
    params = params.merge(avd_map, on="store_point", how="left")
key_order = [
    c for c in [
        "fallback_level",
        "store_point",
        "AvdelingID",
        "month",
        "season",
        "aggregation_window",
        "n_obs",
        "source_period_start",
        "source_period_end",
        "max_source_date",
    ]
    if c in params.columns
]
params = params[key_order + [c for c in params.columns if c not in key_order]]
params.to_parquet(
    PROCESSED / "era5_longrun_climatology_parameters.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy",
)
print(f"Climatology parameters: {len(params):,} group rows")
print(f"  max source date: {params['max_source_date'].max()}  (must be <= {MAX_LOCAL_DATE.date()})")
assert (
    params["max_source_date"].max() <= MAX_LOCAL_DATE
), "Leakage: some climatology rows use dates after 2021-12-31"


## Validation summary

The final checks summarize value ranges, source-date limits, and leakage safety. Notebook 04 consumes the parameter parquet and constructs the climatology-drift centre.


In [ ]:
checks = []
checks.append((
    "temp_hist_mean range (-50,45) C",
    float(big["temp_hist_mean"].between(-50, 45).mean())
    if "temp_hist_mean" in big.columns
    else None,
))
checks.append((
    "humid_hist_mean in [0,100]",
    float(big["humid_hist_mean"].between(0, 100).mean())
    if "humid_hist_mean" in big.columns
    else None,
))
checks.append((
    "cloud_hist_mean in [0,100]",
    float(big["cloud_hist_mean"].between(0, 100).mean())
    if "cloud_hist_mean" in big.columns
    else None,
))
checks.append((
    "wind_hist_mean >= 0",
    float((big["wind_hist_mean"] >= 0).mean())
    if "wind_hist_mean" in big.columns
    else None,
))
checks.append((
    "precip_hist_sum >= 0",
    float((big["precip_hist_sum"] >= 0).mean())
    if "precip_hist_sum" in big.columns
    else None,
))
checks.append((
    "pressure_hist_mean in (900,1100)",
    float(big["pressure_hist_mean"].between(900, 1100).mean())
    if "pressure_hist_mean" in big.columns
    else None,
))
checks.append(("all dates <= 2021-12-31", bool(big["date"].max() <= MAX_LOCAL_DATE)))
for label, value in checks:
    print(f"  {label}: {value}")

print()
print("Output files:")
for f in [
    PROCESSED / "era5_longrun_raw_file_audit.parquet",
    PROCESSED / "era5_longrun_weather_windows.parquet",
    PROCESSED / "era5_longrun_climatology_windows.parquet",
    PROCESSED / "era5_longrun_climatology_parameters.parquet",
    PROCESSED / "era5_longrun_climatology_audit.parquet",
]:
    if f.exists():
        print(f"  {f.relative_to(PROJECT_ROOT)}  ({f.stat().st_size/1_000_000:.2f} MB)")
    else:
        print(f"  MISSING: {f.relative_to(PROJECT_ROOT)}")
